In [1]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model = init_chat_model("openai/gpt-oss-20b", model_provider="groq")
model

d:\agenticai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002587C4F2850>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002587C5B8B50>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [3]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description="Title of the movie")
    year:int=Field(description="Year the movie was released")
    director:str=Field(description="Name of the director of the movie")
    rating:float=Field(description="The movies' ratings out of 10")

In [5]:
structured_model = model.with_structured_output(Movie)
structured_model

RunnableBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002587C4F2850>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002587C5B8B50>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'Title of the movie', 'type': 'string'}, 'year': {'description': 'Year the movie was released', 'type': 'integer'}, 'director': {'description': 'Name of the director of the movie', 'type': 'string'}, 'rating': {'description': "The movies' ratings 

In [6]:
model.invoke("Provide the details of the movie Inception")

AIMessage(content="**Inception (2010)**  \n\n| Category | Details |\n|----------|---------|\n| **Director** | Christopher Nolan |\n| **Screenplay** | Christopher Nolan |\n| **Story** | Christopher Nolan |\n| **Producers** | Christopher Nolan, Emma Thomas |\n| **Production Companies** | Warner Bros. Pictures, Syncopy, Legendary Pictures |\n| **Distributor** | Warner Bros. Pictures |\n| **Release Dates** | • United States: 16 July 2010 (limited)  | • United States: 16 July 2010 (wide) |\n| **Runtime** | 148 minutes |\n| **Country** | United States |\n| **Language** | English |\n| **Budget** | $160\u202fmillion |\n| **Box Office** | $828\u202fmillion worldwide (≈ $2.8\u202fbillion inflation‑adjusted) |\n| **Key Cast** | • Leonardo DiCaprio – Dom Cobb<br>• Joseph Gordon‑Levitt – Arthur<br>• Ellen Page – Ariadne<br>• Tom Hardy – Eames<br>• Ken Watanabe – Saito<br>• Cillian Murphy – Robert Fischer<br>• Marion Cotillard – Mal (Cobb’s wife)<br>• Michael Caine – Professor Stephen Miles |\n| **M

In [7]:
structured_model.invoke("Provide the details of the movie Inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

In [8]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details"""
    title:str=Field(...,description="Title of the movie")
    year:int=Field(...,description="Year the movie was released")
    director:str=Field(...,description="Name of the director of the movie")
    rating:float=Field(...,description="The movies' ratings out of 10")

opt_model = model.with_structured_output(Movie, include_raw=True)  

response = opt_model.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': "We need to provide details about the movie Inception. According to the tool, we can call functions with the movie details. The function signature: Movie({director: string, rating: number, title: string, year: number}). So we can call that function. Let's provide details: director: Christopher Nolan, rating: maybe 8.8 (IMDB). Title: Inception, year: 2010. So we can call the function.", 'tool_calls': [{'id': 'fc_9cb6c1ec-4408-4557-91e9-dc80c395a35f', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 132, 'prompt_tokens': 164, 'total_tokens': 296, 'completion_time': 0.137124831, 'completion_tokens_details': {'reasoning_tokens': 93}, 'prompt_time': 0.008102916, 'prompt_tokens_details': None, 'queue_time': 0.047475249, 'total_time': 0.145227747}, 'model_name': 'openai/g

In [10]:
class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genre:list[str]
    budget:float | None = Field(None, description="Budget in millions in USD")

structured_model = model.with_structured_output(MovieDetails)

response = structured_model.invoke("Provide details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Cillian Murphy', role='Robert Fischer')], genre=['Action', 'Adventure', 'Sci-Fi', 'Thriller'], budget=160000000.0)

In [14]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_typedict=model.with_structured_output(MovieDict)
response=model_typedict.invoke("Please provide the details of the movie Spiderman:Homecoming")
response

{'director': 'Jon Watts',
 'rating': 7.4,
 'title': 'Spider-Man: Homecoming',
 'year': 2017}

In [ ]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_opt = model.with_structured_output(MovieDetails)

response = model_opt.invoke("Provide details about the movie Avengers:Infinity War")
response

{'budget': 316000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'},
  {'name': 'Tom Holland', 'role': 'Peter Parker / Spider-Man'},
  {'name': 'Zoe Saldana', 'role': 'Gamora'},
  {'name': 'Bradley Cooper', 'role': 'Rocket Raccoon'},
  {'name': 'John Krasinski', 'role': 'Happy Hogan'},
  {'name': 'Karen Gillan', 'role': 'Nebula'}],
 'genres': ['Action', 'Adventure', 'Science Fiction'],
 'title': 'Avengers: Infinity War',
 'year': 2018}

In [17]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 32768,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

In [22]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str 
    email: str 
    phone: str 

agent = create_agent(
    model=model,
    response_format=ContactInfo  
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')